# VEC-IoT Phase 3 — MASAC on Google Colab

This notebook runs the upgraded VEC-IoT project in Google Colab.

**What it does**
- uploads the project ZIP
- extracts the project
- installs Python dependencies
- checks the XML traces
- runs a small smoke test
- trains MASAC
- evaluates MASAC against the nearest-edge baseline
- packages the trained model/results for download

> For a quick test, use the small training settings first. Increase the number of episodes/tasks after the pipeline works.

## 1. Upload the project ZIP

Upload the ZIP file produced for this project:

`vec_iot_project_phase3_masac_upgrade.zip`

You can also upload the original `vec_iot_project.zip`, but the Phase-3 MASAC ZIP is the recommended one.

In [ ]:
from google.colab import files
uploaded = files.upload()

project_zip = next((name for name in uploaded if name.endswith(".zip")), None)
assert project_zip is not None, "Please upload a .zip project file."
print("Uploaded:", project_zip)


In [ ]:
import os, shutil, zipfile, pathlib

BASE = "/content"
WORK = os.path.join(BASE, "vec_iot_colab")
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK, exist_ok=True)

with zipfile.ZipFile(os.path.join(BASE, project_zip), "r") as z:
    z.extractall(WORK)

# Find the actual project root (handles ZIPs with an extra top-level folder).
candidates = []
for root, dirs, fs in os.walk(WORK):
    if "train_masac.py" in fs and "src" in dirs:
        candidates.append(root)

assert candidates, "Could not find train_masac.py in the uploaded ZIP."
PROJECT = candidates[0]

print("Project root:", PROJECT)
print("\nFiles:")
for p in sorted(pathlib.Path(PROJECT).glob("*")):
    print(" -", p.name)


## 2. Install dependencies

In [ ]:
import os
os.chdir(PROJECT)
!python -m pip install -q -r requirements_phase3.txt
print("Dependencies installed.")


## 3. Check the XML traces and project imports

In [ ]:
import os
os.chdir(PROJECT)

from src.xml_loader import iter_vehicle_timesteps, iter_task_timesteps

vehicle_first = next(iter_vehicle_timesteps("datasets/vehicles.xml"))
task_first = next(iter_task_timesteps("datasets/tasks.xml"))

print("First vehicle timestamp:", vehicle_first[0])
print("Vehicles at first timestamp:", len(vehicle_first[1]))
print("First task timestamp:", task_first[0])
print("Tasks at first timestamp:", len(task_first[1]))

# Find one task whose creator exists in the vehicle snapshot at the same timestamp.
vehicle_index = {t: v for t, v in iter_vehicle_timesteps("datasets/vehicles.xml")}
aligned = None
for t, tasks in iter_task_timesteps("datasets/tasks.xml"):
    if t in vehicle_index:
        for task in tasks:
            if task.creator in vehicle_index[t]:
                aligned = (t, task, vehicle_index[t][task.creator])
                break
    if aligned:
        break

assert aligned is not None, "No aligned vehicle/task record was found."
print("Aligned timestamp:", aligned[0])
print("Task:", aligned[1].id)
print("Creator:", aligned[1].creator)


## 4. Run a smoke test

In [ ]:
import os
os.chdir(PROJECT)

from src.masac_env import MASACVECEnv
from src.masac import DiscreteMASAC, MASACConfig

t, task, vehicle = aligned
env = MASACVECEnv(vehicle_index[t])

obs, state, mask = env.observation(task, vehicle)
agent = DiscreteMASAC(len(obs), len(state), env.action_dim, MASACConfig(batch_size=16, warmup=16))
agent.set_obs_dim(len(obs))

valid_actions = mask.nonzero()[0]
assert len(valid_actions) > 0, "No valid route/action was generated."

action = agent.act(obs, mask, deterministic=False)
reward, info = env.step_task(task, vehicle, action)

print("Observation dimension:", len(obs))
print("Global state dimension:", len(state))
print("Action dimension:", env.action_dim)
print("Valid actions:", int(mask.sum()))
print("Selected action:", action)
print("Route:", info.get("route"))
print("Latency (s):", info.get("latency_s"))
print("Energy (J):", info.get("energy_j"))
print("Packet loss:", info.get("packet_loss"))
print("Smoke test passed.")


## 5. Quick training run

Start small so you can verify that Colab works.

For a first run:
- `episodes=3`
- `max-tasks=1000`

After that succeeds, use the **full training** cell below.

In [ ]:
import os
os.chdir(PROJECT)

!python train_masac.py     --vehicles datasets/vehicles.xml     --tasks datasets/tasks.xml     --episodes 3     --max-tasks 1000     --save outputs/masac_quick.pt


## 6. Full MASAC training

These are the stronger settings from the project. Training can take a while depending on the Colab runtime.

You can enable a GPU in **Runtime → Change runtime type → T4 GPU**, but the current project is also CPU-compatible.

In [ ]:
import os
os.chdir(PROJECT)

!python train_masac.py     --vehicles datasets/vehicles.xml     --tasks datasets/tasks.xml     --episodes 40     --max-tasks 15000     --save outputs/masac.pt


## 7. Evaluate MASAC vs nearest-edge baseline

In [ ]:
import os
os.chdir(PROJECT)

!python evaluate_masac.py     --vehicles datasets/vehicles.xml     --tasks datasets/tasks.xml     --model outputs/masac.pt     --max-tasks 10000     --csv outputs/evaluation.csv


## 8. Inspect the evaluation CSV

In [ ]:
import os
os.chdir(PROJECT)

import pandas as pd
df = pd.read_csv("outputs/evaluation.csv")
display(df.head())
print("\nRows:", len(df))
print("\nColumns:", list(df.columns))


## 9. Download the trained model and evaluation results

In [ ]:
from google.colab import files
import os

for path in ["outputs/masac.pt", "outputs/evaluation.csv"]:
    full = os.path.join(PROJECT, path)
    if os.path.exists(full):
        files.download(full)
    else:
        print("Not found:", full)


## Optional: save outputs to Google Drive

For long training runs, mount Drive and copy the `outputs/` directory there so your model is not lost when the Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil, os
drive_target = "/content/drive/MyDrive/vec_iot_masac_outputs"
os.makedirs(drive_target, exist_ok=True)

src_outputs = os.path.join(PROJECT, "outputs")
for name in os.listdir(src_outputs):
    src = os.path.join(src_outputs, name)
    dst = os.path.join(drive_target, name)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("Saved outputs to:", drive_target)
